In [2]:
import requests
import pandas as pd
print("Aegis Data collection started")

Aegis Data collection started


In [4]:
import requests

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "PRECTOTCORR,T2M,RH2M,WS2M,PS",
    "community": "AG",
    "longitude": 75.7873,
    "latitude": 26.9124,
    "start": "20250101",
    "end": "20251231",
    "format": "JSON"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

Status Code: 200


In [5]:
data = response.json()
print(data.keys())

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [6]:
weather_data = data["properties"]["parameter"]
print(weather_data.keys())

dict_keys(['PRECTOTCORR', 'T2M', 'RH2M', 'WS2M', 'PS'])


In [7]:
df_jaipur = pd.DataFrame(weather_data)
df_jaipur.index = pd.to_datetime(df_jaipur.index)
df_jaipur = df_jaipur.reset_index()
df_jaipur.rename(columns={"index": "date"}, inplace=True)
df_jaipur.head()

,date,PRECTOTCORR,T2M,RH2M,WS2M,PS
0,2025-01-01,0.0,14.21,45.09,0.92,97.11
1,2025-01-02,0.0,15.32,45.84,0.95,97.08
2,2025-01-03,0.0,16.11,52.02,1.25,97.03
3,2025-01-04,0.0,16.15,55.24,1.78,96.79
4,2025-01-05,0.0,15.62,62.38,1.90,96.69


In [8]:
df_jaipur = df_jaipur.rename(columns={
    "PRECTOTCORR": "rainfall_mm",
    "T2M": "temperature_c",
    "RH2M": "humidity_percent",
    "WS2M": "wind_speed_mps",
    "PS": "pressure_kpa"
})

df_jaipur["location"] = "Jaipur"
df_jaipur["latitude"] = 26.9124
df_jaipur["longitude"] = 75.7873

df_jaipur = df_jaipur[
    [
        "date",
        "location",
        "latitude",
        "longitude",
        "rainfall_mm",
        "temperature_c",
        "humidity_percent",
        "wind_speed_mps",
        "pressure_kpa"
    ]
]

df_jaipur.head()

,date,location,latitude,longitude,rainfall_mm,temperature_c,humidity_percent,wind_speed_mps,pressure_kpa
0,2025-01-01,Jaipur,26.9124,75.7873,0.0,14.21,45.09,0.92,97.11
1,2025-01-02,Jaipur,26.9124,75.7873,0.0,15.32,45.84,0.95,97.08
2,2025-01-03,Jaipur,26.9124,75.7873,0.0,16.11,52.02,1.25,97.03
3,2025-01-04,Jaipur,26.9124,75.7873,0.0,16.15,55.24,1.78,96.79
4,2025-01-05,Jaipur,26.9124,75.7873,0.0,15.62,62.38,1.90,96.69


In [9]:
output_path = "../data/raw/jaipur_weather_2025.csv"

df_jaipur.to_csv(output_path, index=False)

print(f"Data saved successfully: {output_path}")
print(f"Rows: {len(df_jaipur)}")

Data saved successfully: ../data/raw/jaipur_weather_2025.csv
Rows: 365


In [10]:
def fetch_weather_data(location, latitude, longitude, start_date, end_date):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "parameters": "PRECTOTCORR,T2M,RH2M,WS2M,PS",
        "community": "AG",
        "longitude": longitude,
        "latitude": latitude,
        "start": start_date,
        "end": end_date,
        "format": "JSON"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()
    weather_data = data["properties"]["parameter"]

    df = pd.DataFrame(weather_data)

    df.index = pd.to_datetime(df.index)
    df = df.reset_index()
    df.rename(columns={"index": "date"}, inplace=True)

    df = df.rename(columns={
        "PRECTOTCORR": "rainfall_mm",
        "T2M": "temperature_c",
        "RH2M": "humidity_percent",
        "WS2M": "wind_speed_mps",
        "PS": "pressure_kpa"
    })

    df["location"] = location
    df["latitude"] = latitude
    df["longitude"] = longitude

    df = df[
        [
            "date",
            "location",
            "latitude",
            "longitude",
            "rainfall_mm",
            "temperature_c",
            "humidity_percent",
            "wind_speed_mps",
            "pressure_kpa"
        ]
    ]

    return df

In [11]:
test_kota = fetch_weather_data(
    location="Kota",
    latitude=25.2138,
    longitude=75.8648,
    start_date="20250101",
    end_date="20251231"
)

test_kota.head()

,date,location,latitude,longitude,rainfall_mm,temperature_c,humidity_percent,wind_speed_mps,pressure_kpa
0,2025-01-01,Kota,25.2138,75.8648,0.00,14.20,56.58,1.25,97.43
1,2025-01-02,Kota,25.2138,75.8648,0.00,15.98,52.39,1.24,97.39
2,2025-01-03,Kota,25.2138,75.8648,0.00,17.17,52.13,1.35,97.34
3,2025-01-04,Kota,25.2138,75.8648,0.00,18.43,49.96,1.68,97.10
4,2025-01-05,Kota,25.2138,75.8648,2.95,17.98,56.59,1.88,96.99


In [15]:
locations = {
    "Ajmer": (26.4499, 74.6399),
    "Alwar": (27.5530, 76.6346),
    "Balotra": (25.8320, 71.3960),
    "Banswara": (23.5461, 74.4323),
    "Baran": (25.1011, 76.5132),
    "Barmer": (25.7521, 71.3967),
    "Beawar": (26.1010, 74.3203),
    "Bharatpur": (27.2152, 77.5030),
    "Bhilwara": (25.3470, 74.6400),
    "Bikaner": (28.0229, 73.3119),
    "Bundi": (25.4386, 75.6370),
    "Chittorgarh": (24.8887, 74.6269),
    "Churu": (28.2920, 74.9618),
    "Dausa": (26.8932, 76.3375),
    "Deeg": (27.4716, 77.3250),
    "Dholpur": (26.7025, 77.8934),
    "Didwana-Kuchaman": (27.4010, 74.9180),
    "Dungarpur": (23.8431, 73.7147),
    "Hanumangarh": (29.5815, 74.3294),
    "Jaipur": (26.9124, 75.7873),
    "Jaisalmer": (26.9157, 70.9083),
    "Jalore": (25.3500, 72.6167),
    "Jhalawar": (24.5967, 76.1647),
    "Jhunjhunu": (28.1289, 75.3997),
    "Jodhpur": (26.2389, 73.0243),
    "Karauli": (26.4973, 77.0276),
    "Khairthal-Tijara": (27.8000, 76.7000),
    "Kota": (25.2138, 75.8648),
    "Kotputli-Behror": (27.7000, 76.2000),
    "Nagaur": (27.2020, 73.7339),
    "Pali": (25.7711, 73.3234),
    "Phalodi": (27.1310, 72.3683),
    "Pratapgarh": (24.0311, 74.7780),
    "Rajsamand": (25.0715, 73.8798),
    "Sawai Madhopur": (26.0173, 76.3441),
    "Salumbar": (24.1350, 74.0430),
    "Sikar": (27.6094, 75.1399),
    "Sirohi": (24.8850, 72.8580),
    "Sri Ganganagar": (29.9038, 73.8772),
    "Tonk": (26.1664, 75.7885),
    "Udaipur": (24.5854, 73.7125)
}

print(f"Total districts: {len(locations)}")

Total districts: 41


In [16]:
all_weather_data = []

for district, (latitude, longitude) in locations.items():
    print(f"Fetching data for {district}...")

    try:
        df = fetch_weather_data(
            location=district,
            latitude=latitude,
            longitude=longitude,
            start_date="20250101",
            end_date="20251231"
        )

        all_weather_data.append(df)

        print(f"✓ {district} completed — {len(df)} rows")

    except Exception as e:
        print(f"✗ {district} failed — {e}")

print("\n" + "=" * 50)
print("DATA COLLECTION COMPLETED")
print("=" * 50)
print(f"Successful districts: {len(all_weather_data)}")

Fetching data for Ajmer...
✓ Ajmer completed — 365 rows
Fetching data for Alwar...
✓ Alwar completed — 365 rows
Fetching data for Balotra...
✓ Balotra completed — 365 rows
Fetching data for Banswara...
✓ Banswara completed — 365 rows
Fetching data for Baran...
✓ Baran completed — 365 rows
Fetching data for Barmer...
✓ Barmer completed — 365 rows
Fetching data for Beawar...
✓ Beawar completed — 365 rows
Fetching data for Bharatpur...
✓ Bharatpur completed — 365 rows
Fetching data for Bhilwara...
✓ Bhilwara completed — 365 rows
Fetching data for Bikaner...
✓ Bikaner completed — 365 rows
Fetching data for Bundi...
✓ Bundi completed — 365 rows
Fetching data for Chittorgarh...
✓ Chittorgarh completed — 365 rows
Fetching data for Churu...
✓ Churu completed — 365 rows
Fetching data for Dausa...
✓ Dausa completed — 365 rows
Fetching data for Deeg...
✓ Deeg completed — 365 rows
Fetching data for Dholpur...
✓ Dholpur completed — 365 rows
Fetching data for Didwana-Kuchaman...
✓ Didwana-Kuchaman c

In [17]:
df_weather = pd.concat(all_weather_data, ignore_index=True)

print("Total rows:", len(df_weather))
print("Total districts:", df_weather["location"].nunique())

df_weather.head()

Total rows: 14965
Total districts: 41


,date,location,latitude,longitude,rainfall_mm,temperature_c,humidity_percent,wind_speed_mps,pressure_kpa
0,2025-01-01,Ajmer,26.4499,74.6399,0.0,14.73,50.11,1.37,97.36
1,2025-01-02,Ajmer,26.4499,74.6399,0.0,16.67,47.43,1.42,97.31
2,2025-01-03,Ajmer,26.4499,74.6399,0.0,18.04,44.12,1.78,97.23
3,2025-01-04,Ajmer,26.4499,74.6399,0.0,19.34,40.67,2.04,96.97
4,2025-01-05,Ajmer,26.4499,74.6399,0.0,18.04,46.23,1.76,96.93


In [18]:
district_counts = (
    df_weather["location"]
    .value_counts()
    .sort_index()
)

print(district_counts)

location
Ajmer               365
Alwar               365
Balotra             365
Banswara            365
Baran               365
Barmer              365
Beawar              365
Bharatpur           365
Bhilwara            365
Bikaner             365
Bundi               365
Chittorgarh         365
Churu               365
Dausa               365
Deeg                365
Dholpur             365
Didwana-Kuchaman    365
Dungarpur           365
Hanumangarh         365
Jaipur              365
Jaisalmer           365
Jalore              365
Jhalawar            365
Jhunjhunu           365
Jodhpur             365
Karauli             365
Khairthal-Tijara    365
Kota                365
Kotputli-Behror     365
Nagaur              365
Pali                365
Phalodi             365
Pratapgarh          365
Rajsamand           365
Salumbar            365
Sawai Madhopur      365
Sikar               365
Sirohi              365
Sri Ganganagar      365
Tonk                365
Udaipur             365
Name: c

In [19]:
print("\nDistricts:", len(district_counts))
print("Total rows:", district_counts.sum())
print("Minimum rows:", district_counts.min())
print("Maximum rows:", district_counts.max())


Districts: 41
Total rows: 14965
Minimum rows: 365
Maximum rows: 365


In [20]:
missing_values = df_weather.isnull().sum()
print(missing_values)

date                0
location            0
latitude            0
longitude           0
rainfall_mm         0
temperature_c       0
humidity_percent    0
wind_speed_mps      0
pressure_kpa        0
dtype: int64


In [21]:
print("Data Shape:", df_weather.shape)

print("\nData Types:")
print(df_weather.dtypes)

print("\nBasic Statistics:")
display(df_weather.describe())

Data Shape: (14965, 9)

Data Types:
date                datetime64[us]
location                       str
latitude                   float64
longitude                  float64
rainfall_mm                float64
temperature_c              float64
humidity_percent           float64
wind_speed_mps             float64
pressure_kpa               float64
dtype: object

Basic Statistics:


,date,latitude,longitude,rainfall_mm,temperature_c,humidity_percent,wind_speed_mps,pressure_kpa
count,14965,14965.000000,14965.000000,14965.000000,14965.000000,14965.000000,14965.000000,14965.000000
mean,2025-07-02 00:00:00,26.324273,74.733454,2.488441,25.636700,50.946083,2.252158,97.216176
min,2025-01-01 00:00:00,23.546100,70.908300,0.000000,10.420000,5.210000,0.460000,92.960000
25%,2025-04-02 00:00:00,25.213800,73.714700,0.000000,20.090000,28.590000,1.450000,96.300000
50%,2025-07-02 00:00:00,26.238900,74.640000,0.000000,26.510000,49.960000,2.000000,97.280000
75%,2025-10-01 00:00:00,27.401000,76.164700,0.850000,30.260000,74.040000,2.840000,98.230000
max,2025-12-31 00:00:00,29.903800,77.893400,103.130000,41.400000,95.290000,7.650000,100.080000
std,NaN,1.470681,1.693413,6.985746,6.383094,24.868793,1.057250,1.343761


In [23]:
missing_values = df_weather.isnull().sum()

print(missing_values)

date                0
location            0
latitude            0
longitude           0
rainfall_mm         0
temperature_c       0
humidity_percent    0
wind_speed_mps      0
pressure_kpa        0
dtype: int64


In [24]:
duplicates = df_weather.duplicated().sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [25]:
print("Rainfall minimum:", df_weather["rainfall_mm"].min())
print("Rainfall maximum:", df_weather["rainfall_mm"].max())
print("Negative rainfall values:", (df_weather["rainfall_mm"] < 0).sum())

Rainfall minimum: 0.0
Rainfall maximum: 103.13
Negative rainfall values: 0


In [26]:
print("Temperature minimum:", df_weather["temperature_c"].min())
print("Temperature maximum:", df_weather["temperature_c"].max())
print(
    "Temperature outside -10°C to 55°C:",
    ((df_weather["temperature_c"] < -10) |
     (df_weather["temperature_c"] > 55)).sum()
)

Temperature minimum: 10.42
Temperature maximum: 41.4
Temperature outside -10°C to 55°C: 0


In [27]:
print("Humidity minimum:", df_weather["humidity_percent"].min())
print("Humidity maximum:", df_weather["humidity_percent"].max())

print(
    "Invalid humidity values:",
    ((df_weather["humidity_percent"] < 0) |
     (df_weather["humidity_percent"] > 100)).sum()
)

Humidity minimum: 5.21
Humidity maximum: 95.29
Invalid humidity values: 0


In [28]:
print("Wind speed minimum:", df_weather["wind_speed_mps"].min())
print("Wind speed maximum:", df_weather["wind_speed_mps"].max())
print(
    "Negative wind speed values:",
    (df_weather["wind_speed_mps"] < 0).sum()
)

print("\nPressure minimum:", df_weather["pressure_kpa"].min())
print("Pressure maximum:", df_weather["pressure_kpa"].max())
print(
    "Invalid pressure values:",
    (df_weather["pressure_kpa"] <= 0).sum()
)

Wind speed minimum: 0.46
Wind speed maximum: 7.65
Negative wind speed values: 0

Pressure minimum: 92.96
Pressure maximum: 100.08
Invalid pressure values: 0


In [29]:
output_path = "../data/raw/rajasthan_weather_2025.csv"

df_weather.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print(f"File: {output_path}")
print(f"Rows: {len(df_weather)}")
print(f"Columns: {len(df_weather.columns)}")

Dataset saved successfully!
File: ../data/raw/rajasthan_weather_2025.csv
Rows: 14965
Columns: 9
